In [4]:
import mujoco, math, numpy as np
from pathlib import Path

In [5]:
# Settings
MJCF_PATH = Path("../../mechanical/bala-c-plus-simplified/bala-c-plus-simplified.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction

In [6]:
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data = mujoco.MjData(model)
g = float(np.linalg.norm(model.opt.gravity))

In [7]:
def read_a_fwd(d):
    _, accel_y, accel_z = d.sensor("imu_accel").data
    w,x,y,z = d.sensor("imu_orientation").data
    pitch = math.atan2(1.0-2.0*(x*x+y*y), -2.0*(y*z+w*x))
    a_fwd = accel_z - g*math.sin(pitch)      # forward = accel_z, verified
    return a_fwd, pitch, accel_z

In [11]:
# --- Test 1: stationary at several leans -> a_fwd should be ~0 everywhere ---
print("STATIONARY (pose held rigidly, expect a_fwd ~ 0):")
for deg in [0, 5, 11.4, 20, -10]:
    mujoco.mj_resetData(model, data)
    th = math.radians(deg)
    data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
    data.qvel[:] = 0
    mujoco.mj_forward(model, data)          # compute sensors at this exact pose, no stepping
    _, accel_y, accel_z = data.sensor("imu_accel").data
    w,x,y,z = data.sensor("imu_orientation").data
    pitch = math.atan2(1.0-2.0*(x*x+y*y), -2.0*(y*z+w*x))
    a_fwd = accel_z - g*math.sin(pitch)
    print(f"  set {deg:+5.1f}  pitch={math.degrees(pitch):+6.1f}  accel_z={accel_z:+7.3f}  a_fwd={a_fwd:+7.3f}")

STATIONARY (pose held rigidly, expect a_fwd ~ 0):
  set  +0.0  pitch=  +0.0  accel_z= +0.000  a_fwd= +0.000
  set  +5.0  pitch=  -5.0  accel_z= -0.000  a_fwd= +0.855
  set +11.4  pitch= -11.4  accel_z= -0.000  a_fwd= +1.939
  set +20.0  pitch= -20.0  accel_z= -0.000  a_fwd= +3.355
  set -10.0  pitch= +10.0  accel_z= +0.000  a_fwd= -1.703


In [9]:
# --- Test 2: drive forward -> a_fwd should be clearly nonzero, sign matching motion ---
print("\nACCELERATING (drive both wheels, expect a_fwd nonzero):")
mujoco.mj_resetData(model, data)
th = math.radians(11.4)
data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
mujoco.mj_forward(model, data)
lm = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, "left_motor")
rm = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, "right_motor")
for i in range(60):
    data.ctrl[lm] = 0.5
    data.ctrl[rm] = 0.5
    mujoco.mj_step(model, data)
    if i % 10 == 0:
        a_fwd, pitch, az = read_a_fwd(data)
        wv = data.sensor("left_wheel_vel").data[0]
        print(f"  step {i:3d}  a_fwd={a_fwd:+7.3f}  wheel_vel={wv:+6.2f}  pitch={math.degrees(pitch):+6.1f}")


ACCELERATING (drive both wheels, expect a_fwd nonzero):
  step   0  a_fwd= +1.236  wheel_vel= +0.00  pitch= -11.4
  step  10  a_fwd= -0.549  wheel_vel= +3.37  pitch=  -7.7
  step  20  a_fwd= -0.921  wheel_vel= +4.28  pitch=  -0.5
  step  30  a_fwd= -1.597  wheel_vel= +4.56  pitch=  +9.3
  step  40  a_fwd= -2.590  wheel_vel= +4.68  pitch= +22.5
  step  50  a_fwd= -4.043  wheel_vel= +4.76  pitch= +41.2
